In [1]:
import requests
import json
import pandas as pd
import os
from tqdm import tqdm

In [2]:
class whoAmI:
    def __init__(self):
        self.getDim='https://ghoapi.azureedge.net/api/Dimension'
        self.getDimValues='https://ghoapi.azureedge.net/api/DIMENSION/COUNTRY/DimensionValues'
        self.getIndValues='https://ghoapi.azureedge.net/api/Indicator'
        self.getIndFilterVal='https://ghoapi.azureedge.net/api/Indicator?$filter=contains(IndicatorName,'
        self.getIndFilterSpecVal='https://ghoapi.azureedge.net/api/Indicator?$filter=IndicatorName eq '
        self.getIndData='https://ghoapi.azureedge.net/api/'
urls = whoAmI()

In [3]:
DATASET_SAVED = "../../data/raw_official_v2"
URL_INDICATORS = "../../data/urls"

In [4]:
# Khởi tạo thư mục nếu chưa tồn tại
os.makedirs(DATASET_SAVED, exist_ok=True)
os.makedirs(URL_INDICATORS, exist_ok=True)

In [5]:
# Lấy các bộ dữ liệu đã có
existed = set()
for file in os.listdir(DATASET_SAVED):
    indicator = file.split('.')[0]
    existed.add(indicator)

In [6]:
dataset_indicators = set()
for file in os.listdir(URL_INDICATORS):
    with open(URL_INDICATORS + '/' + file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line in existed:
                continue
            dataset_indicators.add(line)
print("Các indicators:", dataset_indicators)

Các indicators: {'SA_0000001823', 'RSUD_29', 'SA_0000001743', 'AIR_15', 'AIR_60', 'RSUD_270', 'BP_05', 'AIR_6', 'SA_0000001746', 'SA_0000001553', 'TAXBEV_ALCOHOLSALESTATUS', 'AIR_44', 'TAXBEV_ABV', 'RS_208', 'NCD_HYP_CONTROL_A', 'NCD_HYP_CONTROL_C', 'AIR_90', 'SA_0000001413', 'SA_0000001843', 'SA_0000001809', 'SA_0000001401', 'SA_0000001414', 'AIR_8', 'SA_0000001400', 'RSUD_660', 'HRH_26', 'BP_04', 'NCD_CCS_DiabetesReg', 'NCDMORT3070', 'WSH_WATER_SAFELY_MANAGED', 'AIR_17', 'SA_0000001697', 'RSUD_68', 'SA_0000001741', 'SA_0000001506', 'SA_0000001740', 'NCD_CCS_riskstrat', 'SA_0000001409', 'NCD_BMI_PLUS2C', 'AIR_16', 'NCD_GLUC_01', 'NCD_CCS_ALC_TARGET', 'NCD_CCS_rheumfollowup', 'NCD_GLUC_02', 'SA_0000001705', 'SA_0000001747', 'SA_0000001398', 'SA_0000001813', 'RSUD_740', 'SA_0000001801', 'SA_0000001475', 'RSUD_85', 'SA_0000001520', 'NCD_CCS_BP_TARGET', 'NCD_BMI_25A', 'AIR_71', 'SA_0000001839', 'SA_0000001807_AA', 'RSUD_3', 'NCD_BMI_MEANC', 'RSUD_180', 'NCD_DIABETES_PREVALENCE_CRUDE', 'SD

In [7]:
def request_data(indicator):
    resp = requests.get(urls.getIndData + '/' + indicator)
    data = json.loads(resp.content)["value"]
    fields = ['ParentLocationCode', 'SpatialDim', 'Value', 'NumericValue', 'TimeDimensionBegin', 'TimeDimensionEnd', 'TimeDimensionValue', 'TimeDimType', 'TimeDim', 'IndicatorCode']

    records = []
    for row in data:
        _tempo = dict()
        for field in fields:
            _tempo[field] = row[field]
        records.append(_tempo)
    
    with open(DATASET_SAVED + '/' + indicator + '.json', "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

In [8]:
# Thử chương trình lấy một bộ dữ liệu
request_data("SA_0000001403")

In [9]:
# Hàm lấy hết dữ liệu
def retrieve_all():
    for indicator in tqdm(dataset_indicators):
        try:
            request_data(indicator)
        except Exception as e:
            print("Error:", e)

In [10]:
retrieve_all()

 56%|█████▌    | 159/284 [14:02<03:33,  1.71s/it] 

Error: Expecting value: line 1 column 1 (char 0)


100%|██████████| 284/284 [22:30<00:00,  4.76s/it]


In [11]:
# Biến chỉ định lưu trữ
CONCAT_GROUP = "../../data/raw_concat"
os.makedirs(CONCAT_GROUP, exist_ok=True)

In [12]:
# Tiến hành tổ chức và nhóm lại dữ liệu vào file gốc theo các nhãn
def prepare_data_through_group(group):
    # Group là nhãn dữ liệu, tên thư mục
    target = group.split('.')[0]
    objectives = set()

    with open(URL_INDICATORS + '/' + group, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            objectives.add(line)
    
    array_data = []
    for file in tqdm(os.listdir(DATASET_SAVED)):
        _label = file.split('.')[0]
        if _label in objectives:
            # Đúng nhãn dữ liệu, tiến hành load ra và ghép lại thành một file hoàn chỉnh
            with open(DATASET_SAVED + '/' + file, "rb") as f:
                data = json.load(f)
                
                # Tiến hành nối dữ liệu vào file
            array_data = array_data + data
                
    concated_result = CONCAT_GROUP + '/' + target + '.json'
    
    print("Tổng số điểm dữ liệu là:", len(array_data))
    with open(concated_result, "w", encoding="utf-8") as f:
        json.dump(array_data, f, ensure_ascii=False, indent=2)
    print("Hoàn thành xử lý:", target)

In [13]:
# Chạy xử lí chỉ số BMI
prepare_data_through_group("BMI.txt")

100%|██████████| 286/286 [00:02<00:00, 142.48it/s]


Tổng số điểm dữ liệu là: 453077
Hoàn thành xử lý: BMI


In [14]:
# CHạy xử lí chỉ số Diabetes
prepare_data_through_group("diabetes.txt")

100%|██████████| 286/286 [00:01<00:00, 182.97it/s]


Tổng số điểm dữ liệu là: 211564
Hoàn thành xử lý: diabetes


In [15]:
# Chạy xử lí tiêu thụ cồn
prepare_data_through_group("alcohol_consumption.txt")

100%|██████████| 286/286 [00:09<00:00, 31.08it/s] 


Tổng số điểm dữ liệu là: 301920
Hoàn thành xử lý: alcohol_consumption


In [16]:
# Chạy xử lí air_pollution
prepare_data_through_group("air_pollution.txt")

100%|██████████| 286/286 [00:03<00:00, 89.48it/s] 


Tổng số điểm dữ liệu là: 491076
Hoàn thành xử lý: air_pollution


In [17]:
# Chạy xử lí cholesterol
prepare_data_through_group("cholesterol.txt")

100%|██████████| 286/286 [00:00<00:00, 928.34it/s]


Tổng số điểm dữ liệu là: 71096
Hoàn thành xử lý: cholesterol


In [18]:
# Chạy xử lí Glucose
prepare_data_through_group("glucose.txt")

100%|██████████| 286/286 [00:00<00:00, 2899.34it/s]

Tổng số điểm dữ liệu là: 35070


Hoàn thành xử lý: glucose


In [19]:
# Chạy xử lí hoạt động thể chất
prepare_data_through_group("physical_activities.txt")

100%|██████████| 286/286 [00:00<00:00, 1311.47it/s]


Tổng số điểm dữ liệu là: 35523
Hoàn thành xử lý: physical_activities


In [20]:
# Chạy xử lí tiếp cận cơ sở y tế
prepare_data_through_group("infrastructure.txt")

100%|██████████| 286/286 [00:00<00:00, 1158.26it/s]


Tổng số điểm dữ liệu là: 8989
Hoàn thành xử lý: infrastructure


In [21]:
# Chạy xử lí thuốc lá
prepare_data_through_group("tobacco.txt")

100%|██████████| 286/286 [00:00<00:00, 1349.06it/s]


Tổng số điểm dữ liệu là: 25101
Hoàn thành xử lý: tobacco


In [22]:
# Chạy xử lí bệnh tim mạch
prepare_data_through_group("cardiovascular_diseases.txt")

100%|██████████| 286/286 [00:02<00:00, 132.85it/s]


Tổng số điểm dữ liệu là: 236876
Hoàn thành xử lý: cardiovascular_diseases
